# Object Detection Export Example

This notebook provides an end-to-end example on how to train and export an object detection model to a format accepted by uniVision Module Image ONNX. This will allow you to deploy your models on the Wenglor's B60 Smart Camera and MVC Machine Vision Controller.

DISCLAIMER: Focus of this notebook is to provide an example for the export and quantization. The training code is provided purely for illustration and not meant to provide a production-grade model. You will want to optimize the training pipeline and add more images. Depending on your situation, you can start from one of the following steps: 
1. Train PyTorch model.
    * If you don't have a trained model, start here. 
2. Export PyTorch model to ONNX.
    * If you already have a trained PyTorch model, start here.
3. Quantize ONNX model
    * If you have a trained ONNX model and it supports quantization, then start here. For this step, you will also need to provide a dataset that your model had been trained on. This dataset will be used for a calibration step during model quantization. 
4. Export ONNX model.
    * If your ONNX model does not support quantization or you cannot provide a calibration dataset, begin here. 
  
The models below have been tested and are known to run successfully:
* mmdetection YOLOX-tiny
* mmdetection YOLOX-S

We recommend a maximum input image size of 480×480 to stay within the memory and latency constraints of the B60 Smart Camera.

**What is quantization and why do we need it?**

Quantization is a technique used to make machine learning models run faster and more efficiently, especially on smaller devices with limited resources, like smartphones or smart cameras. By reducing the precision of the numbers the model uses, it can significantly decrease the amount of memory and computing power required, without losing too much accuracy. For example, our benchmark results show that a quantized ResNet50 model running on a B60 camera achieves an inference speed of 50 milliseconds, which is 8.4 times faster than the unquantized version (420 milliseconds). 

## 1. Train PyTorch model 

In [ ]:
# https://github.com/pytorch/pytorch/issues/140765
import getpass
import os

# Add dummy username so default_cache_dir() doesn't blow up
getpass.getuser = lambda: "torchuser"

# Also set the cache dir explicitly
os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/tmp/torchinductor_cache"

In [ ]:
import copy
import glob
import json
import os
import random
import shutil
import time
import uuid
from functools import partial
from pathlib import Path

import cv2
import datumaro as dm
import matplotlib.pyplot as plt
import mmcv
import mmdet
import numpy as np
import onnxruntime as ort
import pandas as pd
import requests
import torch
import torch.nn as nn
from mmdet.apis import inference_detector, init_detector
from mmdet.visualization import DetLocalVisualizer
from mmengine.config import Config
from mmengine.runner import Runner
from onnxruntime.quantization.quantize import (
    CalibrationDataReader,
    CalibrationMethod,
    QuantFormat,
    QuantType,
    quantize_static,
)
from onnxruntime.quantization.shape_inference import quant_pre_process
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from sklearn.model_selection import train_test_split
from torchvision.ops import nms
from utils.bbox import visualize_bbox
from utils.constants import IMAGENET_MEAN, IMAGENET_STD
from utils.enums import DatasetColorMode
from utils.export import export_univision_model_v3
from utils.image import (
    ensure_3ch_image,
    get_image_size,
    is_rgb_image,
    read_and_resize_input_example,
    read_image_file,
)
from utils.quantization import (
    TorchCalibrationDataReader,
    find_postprocess_nodes_to_exclude,
)

In [ ]:
AI_INPUT_IMAGE_MAX_DIMENSION = 416
assert (
    AI_INPUT_IMAGE_MAX_DIMENSION % 32 == 0
), "AI_INPUT_IMAGE_SIZE must be divisible by 32"

# threshold to use to decide if a class probability enables the relative class
CLASS_THRESHOLD = 0.5

DATA_ROOT = Path("../data")
ANNOTATION_FOLDER = DATA_ROOT / "coco-annotations/"

EXAMPLE_IMAGE_FOLDER = DATA_ROOT / "images/multi-label/"
MODEL_FOLDER = DATA_ROOT / "model"

MMDETECTION_DATA_FOLDER = DATA_ROOT / "mmdetection"
MMDETECTION_OUTPUT_FOLDER = DATA_ROOT / "work_dirs" / "yolox_custom"

MMDETECTION_DATA_FOLDER.mkdir(exist_ok=True)
MMDETECTION_OUTPUT_FOLDER.mkdir(exist_ok=True, parents=True)
MODEL_FOLDER.mkdir(parents=True, exist_ok=True)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"The training will use {device} device")

In [ ]:
# this is to get around a change in pytorch default settings
torch.load = partial(torch.load, weights_only=False)

### 1.1 Prepare dataset

For this example, we will use a small dataset with some images acquired from a desk. Each image could contain one or more of these office items: a pencil, a sharpener and a adhesive tape. The trained model will be able to detect the presence of these office items. The images are stored in different folders in the `EXAMPLE_IMAGE_FOLDER` base path: the name of each folder describes the classes of the images contained. For example, the folder `sharpener-pencil` contains all the images that show a sharpener and a pencil. 

This object detection example uses a subset of the images from the multilabel dataset. We will first read it with Datumaro, then we will copy the images from our multilabel dataset to the COCO dataset format. The datumaro library supports a number of formats, for more details please visit this [link](https://open-edge-platform.github.io/datumaro/stable/docs/jupyter_notebook_examples/notebooks/14_import_export_det_data.html).

In [ ]:
coco_dataset = dm.Dataset.import_from(
    path=ANNOTATION_FOLDER,
    format="coco_instances",
)

In [ ]:
index_to_class = {v: k for k, v in coco_dataset.categories()[1]._indices.items()}
classes = list(index_to_class.values())
classes

In [ ]:
# For custom dataset, please skip this
# Copy images from our multilabel dataset to the coco dataset format
if ANNOTATION_FOLDER == DATA_ROOT / "coco-annotations/":
    image_names = [Path(item.media.path).name for item in coco_dataset]
    image_source_paths = [
        next(EXAMPLE_IMAGE_FOLDER.rglob(image_name)) for image_name in image_names
    ]
    for image_source_path in image_source_paths:
        shutil.copy2(
            image_source_path, ANNOTATION_FOLDER / "images" / image_source_path.name
        )
else:
    print("Skipping image copy due to custom image dataset")

### 1.2 Visualize bounding boxes

Let's check the annotation to ensure that the dataset is read correctly.

In [ ]:
coco_dataset_list = list(coco_dataset)
datumaro_bbox_folder = MMDETECTION_OUTPUT_FOLDER / "datumaro_bbox"
datumaro_bbox_folder.mkdir(exist_ok=True)

for item in coco_dataset_list:
    image_path = Path(item.media.path)

    vis_image = visualize_bbox(
        image_path=image_path,
        boxes=np.array([annotation.points for annotation in item.annotations]),
        labels=np.array([annotation.label for annotation in item.annotations]),
        scores=np.array([1 for _ in item.annotations]),
        class_names=classes,
        score_threshold=0.5,
        save_path=datumaro_bbox_folder / image_path.with_suffix(".jpg").name,
    )
print(f"Datumaro bounding boxes were visualized and saved to {datumaro_bbox_folder}")

### 1.3 Train-test-split

It is important to ensure that classes are equally distributed in train and validation. We will concatenate the class names so that we can do a stratified train-test-split. mmdetection expects the folder to be in the COCO format, we will transform our dataset to fit it.

```
dataset/
├── train/
│   ├── annotations/
│   │   └── instances_default.json
│   └── images/default/
│       ├── image1.jpg
│       ├── image2.jpg
│       └── ... (other image files)
└── valid/
    ├── annotations/
    │   └── instances_default.json
    └── images/default/
        ├── image1.jpg
        ├── image2.jpg
        └── ... (other image files)
```

In [ ]:
# Iterate over every image/item in the dataset
data = {}
for item in coco_dataset:
    image_id = item.id  # Usually the filename stem (without extension)
    # Get all annotations (bounding boxes) for this image
    label_counts = {}
    for ann in item.annotations:
        if ann.type == dm.AnnotationType.bbox:  # Ensure it's a bounding box
            label_name = index_to_class[ann.label]
            label_counts[label_name] = label_counts.get(label_name, 0) + 1

    # Store in the main dict
    data[image_id] = label_counts

df = pd.DataFrame.from_dict(data, orient="index").fillna(0).astype(int)
column_names = list(df.columns)

missing_class = [class_i for class_i in classes if class_i not in column_names]
if missing_class:
    print(
        f"Warning! {missing_class} are missing but are included in the annotation categories!"
    )

df = df.reset_index(names="image_id")
df["label_combo"] = df[column_names].apply(
    lambda row: "".join(str(bool(x)) for x in row), axis=1
)

df.head(3)

In [ ]:
df[["label_combo"]].value_counts()

In [ ]:
train_image_id, valid_image_id = train_test_split(
    df.image_id.values,
    test_size=0.2,
    random_state=0,
    stratify=df.label_combo,
)
train_image_id.shape, valid_image_id.shape

In [ ]:
train = copy.deepcopy(coco_dataset)
valid = copy.deepcopy(coco_dataset)

for image_id in valid_image_id:
    train.remove(image_id)

for image_id in train_image_id:
    valid.remove(image_id)

In [ ]:
train.export("train", format="coco_instances", save_media=True)
valid.export("valid", format="coco_instances", save_media=True)

In [ ]:
for phase in ["train", "valid"]:
    folder_path = MMDETECTION_DATA_FOLDER / phase
    if folder_path.exists():
        print(f"{folder_path} exists. Copy is skipped")
    else:
        Path(phase).rename(folder_path)

In [ ]:
# Fix duplicate annotation IDs produced by Datumaro export
for phase in ["train", "valid"]:
    ann_path = (
        MMDETECTION_DATA_FOLDER / phase / "annotations" / "instances_default.json"
    )
    with open(ann_path) as f:
        coco_json = json.load(f)
    for new_id, ann in enumerate(coco_json["annotations"], start=1):
        ann["id"] = new_id
    with open(ann_path, "w") as f:
        json.dump(coco_json, f)
    print(f"{phase}: re-numbered {len(coco_json['annotations'])} annotation IDs")

### 1.5 Find image size with minimal padding

The accuracy of bounding box tends to improve when the image is not distorted, we will therefore resize the image while preserving the aspect ratio, the AI input image size will be achieved by padding. The following section will find the maximum image size of this dataset and calculate the image size with minimal padding based on the specified `AI_INPUT_IMAGE_MAX_DIMENSION` for optimal inference and training time. 

In [ ]:
image_sizes = []
for item in coco_dataset:
    image_sizes.append(get_image_size(item.media.path))
print(f"unique image sizes: {set(image_sizes)}")
image_sizes = np.array(image_sizes)

max_image_size = (np.max(image_sizes[:, 0]), np.max(image_sizes[:, 1]))

In [ ]:
def get_image_size_with_minimal_padding(
    original_size: tuple, max_dimension: int
) -> tuple:
    """
    Get image size with minimal padding that is divisible by 32 and has max_dimension as one dimension.

    Args:
        original_size: Original (width, height) tuple
        max_dimension: Maximum allowed dimension (e.g., 640, 512, 768)

    Returns:
        New (width, height) tuple that is divisible by 32 and minimizes padding
    """
    if max_dimension % 32 != 0:
        raise ValueError("Max dimension must be divisible by 32.")
    orig_height, orig_width = original_size
    aspect_ratio = orig_width / orig_height

    # Determine which dimension should be the max dimension
    if orig_height > orig_width:
        new_height = max_dimension
        new_width = int(np.ceil(new_height * aspect_ratio / 32) * 32)
    else:
        new_width = max_dimension
        new_height = int(np.ceil(new_width / aspect_ratio / 32) * 32)

    return (new_height, new_width)

In [ ]:
AI_INPUT_IMAGE_SIZE = get_image_size_with_minimal_padding(
    max_image_size, AI_INPUT_IMAGE_MAX_DIMENSION
)
print(f"AI_INPUT_IMAGE_SIZE: {AI_INPUT_IMAGE_SIZE}")

### 1.6 Train model

mmdetection trains models with a configuration based, for more details click [here](https://mmdetection.readthedocs.io/en/dev-3.x/user_guides/config.html). The following configuration works well for YOLOX-S. Change `MODEL_NAME` to use a model specifified in `YOLOX_MODELS`

In [ ]:
YOLOX_MODELS = {
    "yolox_tiny": {
        "config": "yolox_tiny_8xb8-300e_coco.py",
        "checkpoint": "yolox_tiny_8x8_300e_coco_20211124_171234-b4047906.pth",
        "url": "https://download.openmmlab.com/mmdetection/v2.0/yolox/yolox_tiny_8x8_300e_coco/yolox_tiny_8x8_300e_coco_20211124_171234-b4047906.pth",
    },
    "yolox_s": {
        "config": "yolox_s_8xb8-300e_coco.py",
        "checkpoint": "yolox_s_8x8_300e_coco_20211121_095711-4592a793.pth",
        "url": "https://download.openmmlab.com/mmdetection/v2.0/yolox/yolox_s_8x8_300e_coco/yolox_s_8x8_300e_coco_20211121_095711-4592a793.pth",
    },
}

MODEL_NAME = "yolox_s"  # change this to switch models

model_cfg = YOLOX_MODELS[MODEL_NAME]
default_model_config_path = Path(f"./mmdetection/configs/yolox/{model_cfg['config']}")
pretrained_checkpoint_path = MODEL_FOLDER / model_cfg["checkpoint"]

In [ ]:
if not pretrained_checkpoint_path.exists():
    print(f"Downloading checkpoint")
    with requests.get(model_cfg["url"], stream=True) as response:
        response.raise_for_status()  # Raises an exception for bad status codes (4xx/5xx)

        with open(pretrained_checkpoint_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:  # Filter out keep-alive chunks
                    f.write(chunk)
    print(f"Checkpoint downloaded successfully to {pretrained_checkpoint_path}")
else:
    print(f"Checkpoint already exists, skipping download")

In [ ]:
num_train_images = len(train_image_id)

BATCH_SIZE = min(8, num_train_images // 4)
steps_per_epoch = num_train_images // BATCH_SIZE

# Target ~2500 total training steps for finetuning
target_total_steps = 2500
EPOCHS = max(60, target_total_steps // steps_per_epoch)

num_last_epochs = max(10, int(EPOCHS * 0.5))

# Disable MixUp for very small datasets — not enough diversity to benefit
use_mixup = num_train_images >= 500

# Linear scaling rule
base_lr = 0.001
effective_lr = base_lr * (BATCH_SIZE / 64)

weight_decay = 0.1  # 0.05

backbone_lr_ratio = 0.5
neck_lr_ratio = 0.7
head_lr_ratio = 1.0

print(f"Dataset size:      {num_train_images} images")
print(f"Batch size:        {BATCH_SIZE}")
print(f"Steps per epoch:   {steps_per_epoch}")
print(f"Epochs:            {EPOCHS}")
print(f"  - with aug:      {EPOCHS - num_last_epochs}")
print(f"  - without aug:   {num_last_epochs}")
print(f"MixUp enabled:     {use_mixup}")
print(f"Total steps:       {EPOCHS * steps_per_epoch}")
print(f"Effective LR:      {effective_lr}")
print(f"Weight decay:      {weight_decay}")
print(f"Backbone LR ratio: {backbone_lr_ratio}")
print(f"Neck LR ratio:     {neck_lr_ratio}")
print(f"Head LR ratio:     {head_lr_ratio}")

In [ ]:
img_scale = AI_INPUT_IMAGE_SIZE
annotation_json_relative_path = Path("annotations/instances_default.json")
images_relative_path = Path("images/default/")
data_root = str(MMDETECTION_DATA_FOLDER)

In [ ]:
cfg = Config.fromfile(str(default_model_config_path))
cfg.data_root = data_root
cfg.dataset_type = "CocoDataset"
cfg.backend_args = None
cfg.base_lr = base_lr

# --- Pipeline  ---
train_pipeline_steps = [
    dict(type="Mosaic", img_scale=img_scale, pad_val=114.0),
    dict(
        type="RandomAffine",
        scaling_ratio_range=(0.5, 1.5),
        border=(-img_scale[0] // 2, -img_scale[1] // 2),
    ),
]
if use_mixup:
    train_pipeline_steps.append(
        dict(type="MixUp", img_scale=img_scale, ratio_range=(0.8, 1.6), pad_val=114.0)
    )
train_pipeline_steps.extend(
    [
        dict(type="YOLOXHSVRandomAug"),
        dict(type="RandomFlip", prob=0.5),
        dict(type="Resize", scale=img_scale, keep_ratio=True),
        dict(
            type="Pad",
            size_divisor=32,  # Keep 32-pixel constraint
            pad_to_square=False,  # KEY: Don't force square!
            pad_val=dict(img=(114.0, 114.0, 114.0)),
        ),
        dict(type="FilterAnnotations", min_gt_bbox_wh=(1, 1), keep_empty=False),
        dict(type="PackDetInputs"),
    ]
)
cfg.train_pipeline = train_pipeline_steps

cfg.train_dataset = dict(
    type="MultiImageMixDataset",
    dataset=dict(
        type=cfg.dataset_type,
        data_root=str(data_root),
        ann_file=str(Path("train") / annotation_json_relative_path),
        data_prefix=dict(img=str(Path("train") / images_relative_path)),
        metainfo=dict(classes=classes),
        pipeline=[
            dict(type="LoadImageFromFile", backend_args=cfg.backend_args),
            dict(type="LoadAnnotations", with_bbox=True),
        ],
        filter_cfg=dict(filter_empty_gt=False, min_size=32),
        backend_args=cfg.backend_args,
    ),
    pipeline=cfg.train_pipeline,
)

cfg.test_pipeline = [
    dict(type="LoadImageFromFile", backend_args=cfg.backend_args),
    dict(type="Resize", scale=img_scale, keep_ratio=True),
    dict(
        type="Pad",
        size_divisor=32,  # Keep 32-pixel constraint
        pad_to_square=False,  # KEY: Don't force square!
        pad_val=dict(img=(114.0, 114.0, 114.0)),
    ),
    dict(type="LoadAnnotations", with_bbox=True),
    dict(
        type="PackDetInputs",
        meta_keys=("img_id", "img_path", "ori_shape", "img_shape", "scale_factor"),
    ),
]

cfg.train_dataloader = dict(
    batch_size=BATCH_SIZE,
    num_workers=4,
    persistent_workers=True,
    sampler=dict(type="DefaultSampler", shuffle=True),
    dataset=cfg.train_dataset,
)

cfg.val_dataloader = dict(
    batch_size=BATCH_SIZE,
    num_workers=4,
    persistent_workers=True,
    drop_last=False,
    sampler=dict(type="DefaultSampler", shuffle=False),
    dataset=dict(
        type=cfg.dataset_type,
        data_root=str(data_root),
        ann_file=str(Path("valid") / annotation_json_relative_path),
        data_prefix=dict(img=str(Path("valid") / images_relative_path)),
        test_mode=True,
        metainfo=dict(classes=classes),
        pipeline=cfg.test_pipeline,
        backend_args=cfg.backend_args,
    ),
)
cfg.test_dataloader = cfg.val_dataloader

cfg.val_evaluator = dict(
    type="CocoMetric",
    ann_file=str(Path(data_root) / Path("valid") / annotation_json_relative_path),
    metric="bbox",
    backend_args=cfg.backend_args,
)
cfg.test_evaluator = cfg.val_evaluator

cfg.model.bbox_head.num_classes = len(classes)

# --- Training schedule ---
cfg.train_cfg.max_epochs = EPOCHS
cfg.train_cfg.val_interval = 1

# --- Optimizer ---
cfg.optim_wrapper = dict(
    type="AmpOptimWrapper",
    optimizer=dict(type="AdamW", lr=effective_lr, weight_decay=weight_decay),
    paramwise_cfg=dict(
        custom_keys={
            "backbone": dict(lr_mult=backbone_lr_ratio, decay_mult=1.0),
            "neck": dict(lr_mult=neck_lr_ratio, decay_mult=1.0),
            "bbox_head": dict(lr_mult=head_lr_ratio, decay_mult=1.0),
        }
    ),
    clip_grad=dict(max_norm=10.0, norm_type=2),
)

# --- LR schedulers ---
cfg.param_scheduler = [
    dict(
        type="mmdet.QuadraticWarmupLR",
        by_epoch=True,
        begin=0,
        end=3,
        convert_to_iter_based=True,
    ),
    dict(
        type="CosineAnnealingLR",
        eta_min=effective_lr * 0.05,
        begin=3,
        T_max=EPOCHS - num_last_epochs,
        end=EPOCHS - num_last_epochs,
        by_epoch=True,
        convert_to_iter_based=True,
    ),
    dict(
        type="CosineAnnealingLR",
        eta_min=effective_lr * 0.01,
        begin=EPOCHS - num_last_epochs,
        T_max=num_last_epochs,
        end=EPOCHS,
        by_epoch=True,
        convert_to_iter_based=True,
    ),
]

# --- Custom hooks ---
cfg.custom_hooks = [
    dict(
        type="YOLOXModeSwitchHook",
        num_last_epochs=num_last_epochs,
        priority=48,
    ),
    dict(type="SyncNormHook", priority=48),
    dict(
        type="EMAHook",
        ema_type="ExpMomentumEMA",
        momentum=0.0001,
        update_buffers=True,
        priority=49,
    ),
]

cfg.default_hooks.logger.interval = 50
cfg.default_hooks.checkpoint.interval = 5
cfg.default_hooks.checkpoint.save_best = "coco/bbox_mAP"
cfg.default_hooks.checkpoint.rule = "greater"

cfg.work_dir = str(MMDETECTION_OUTPUT_FOLDER)
cfg.load_from = str(pretrained_checkpoint_path)
cfg.resume = False

In [ ]:
runner = Runner.from_cfg(cfg)
runner.train()

### 1.7 Evaluate results

In [ ]:
config_file_path = MMDETECTION_OUTPUT_FOLDER / default_model_config_path.name
checkpoint_file_path = sorted(
    list(MMDETECTION_OUTPUT_FOLDER.glob("best_coco_bbox_mAP_epoch_*"))
)[-1]
model = init_detector(str(config_file_path), str(checkpoint_file_path), device="cpu")
model.eval()
print("Model loaded")

In [ ]:
valid_images_dir = MMDETECTION_DATA_FOLDER / "valid" / "images" / "default"

valid_image_paths = [
    p
    for p in valid_images_dir.iterdir()
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".avif"}
]

In [ ]:
raw_predictions_folder = MMDETECTION_OUTPUT_FOLDER / "raw_prediction"
raw_predictions_folder.mkdir(exist_ok=True)

for image_path in valid_image_paths:
    result = inference_detector(model, image_path)

    visualizer = DetLocalVisualizer()
    visualizer.dataset_meta = model.dataset_meta

    img = mmcv.imread(
        image_path,
    )
    visualizer.add_datasample(
        name="result", image=img, data_sample=result, draw_gt=False, show=False
    )

    vis_img = visualizer.get_image()
    cv2.imwrite(raw_predictions_folder / image_path.with_suffix(".jpg").name, vis_img)
print(f"Raw predictions were visualized and saved to {raw_predictions_folder}")

## 2. Export PyTorch Model to ONNX

If you already have a trained PyTorch model, start here.

**Note:** Quantizing for **NPU** acceleration may fail if your model contains unsupported layers/operators.  
Only the models listed at the start of this notebook are guaranteed to work in the full NPU workflow.

Alternatively, you can quantize and run on **CPU** instead. This still gives significant speed-ups with much broader layer support. See `inference_device` in [metadata.md](../metadata.md), this can be specifed at the final step of this notebook.

The module image ONNX expects the ONNX model output to be as follows:
* boxes: The shape of the corresponding output must be `(detections_count, 4)`  , the type is float32. The meaning of these 4 numbers is described by `boxes_format` and `boxes_coordinates`.
* labels: The shape of the corresponding output must be `(detections_count,)`, the type is int64. Each value must be in the range `[0..classes-1]`.
* scores: The shape of the corresponding output must be `(detections_count,)`, the type is float32. Each value must be in the range `[0..1]`.

In this example, we will create a wrapper that post-processes the output of the ONNX model to follow the above specifications.



In [ ]:
FP32_ONNX_MODEL_PATH = os.path.join(MODEL_FOLDER, "fp32_model.onnx")

In [ ]:
class YOLOXPostprocessExport(nn.Module):
    """
    YOLOX export wrapper replicating mmdetection's bbox_head.predict() logic.
    Includes NMS so it is traced into the ONNX graph.
    """

    def __init__(
        self,
        base_model: nn.Module,
        img_size,
        conf_thre: float = 0.01,
        nms_thre: float = 0.45,
    ):
        super().__init__()
        self.backbone = base_model.backbone
        self.neck = base_model.neck
        self.bbox_head = base_model.bbox_head
        self.img_size = img_size
        self.conf_thre = conf_thre
        self.nms_thre = nms_thre
        self.num_classes = self.bbox_head.cls_out_channels

        self.backbone.eval()
        self.neck.eval()
        self.bbox_head.eval()

    def _flatten_head_outputs(self, cls_scores, bbox_preds, objectnesses):
        """Flatten multi-scale head outputs and concatenate."""

        def _flat(tensors, last_dim):
            return torch.cat(
                [t.permute(0, 2, 3, 1).reshape(1, -1, last_dim) for t in tensors],
                dim=1,
            )

        return (
            _flat(cls_scores, self.num_classes).sigmoid(),
            _flat(bbox_preds, 4),
            _flat(objectnesses, 1).squeeze(-1).sigmoid(),
        )

    def _bbox_decode(self, priors, bbox_preds):
        """Decode bbox predictions using grid priors (replicates YOLOXHead._bbox_decode)."""
        xy = bbox_preds[..., :2] * priors[:, 2:] + priors[:, :2]
        wh = bbox_preds[..., 2:].exp() * priors[:, 2:]
        return torch.cat([xy - wh / 2, xy + wh / 2], dim=-1)

    def forward(self, x):
        feat = self.neck(self.backbone(x))
        cls_scores, bbox_preds, objectnesses = self.bbox_head.forward(feat)

        # Generate grid priors
        featmap_sizes = [cs.shape[2:] for cs in cls_scores]
        priors = torch.cat(
            self.bbox_head.prior_generator.grid_priors(
                featmap_sizes,
                dtype=cls_scores[0].dtype,
                device=cls_scores[0].device,
                with_stride=True,
            )
        )

        # Flatten multi-scale outputs → (1, N, C)
        cls_scores, bbox_preds, obj_scores = self._flatten_head_outputs(
            cls_scores,
            bbox_preds,
            objectnesses,
        )

        # Decode boxes and compute final scores
        boxes = self._bbox_decode(priors, bbox_preds)[0]
        max_cls_scores, labels = cls_scores[0].max(dim=1)
        scores = max_cls_scores * obj_scores[0]

        # Filter by confidence
        keep = scores >= self.conf_thre
        boxes, labels, scores = boxes[keep], labels[keep], scores[keep]

        keep = nms(boxes, scores, self.nms_thre)
        boxes, labels, scores = boxes[keep], labels[keep], scores[keep]

        return boxes, labels, scores

In [ ]:
wrapper = YOLOXPostprocessExport(
    base_model=model,
    img_size=AI_INPUT_IMAGE_SIZE,
    conf_thre=0.01,
    nms_thre=0.45,
)

wrapper.eval()

# Export to ONNX
dummy_input = torch.randn(1, 3, AI_INPUT_IMAGE_SIZE[0], AI_INPUT_IMAGE_SIZE[1])
torch.onnx.export(
    wrapper,
    dummy_input,
    FP32_ONNX_MODEL_PATH,
    input_names=["input"],
    output_names=["boxes", "labels", "scores"],
    opset_version=20,
    do_constant_folding=True,
    dynamo=True,
    external_data=False,
    optimize=True,
)
print(f"ONNX FP32 model exported to {FP32_ONNX_MODEL_PATH}")

### 2.1 Evaluate the ONNX FP32 model

We will use the exported model to predict the validation dataset to verify the correctness of our wrapper.

In [ ]:
def preprocess_img_yolox(image, target_size):
    """
    Preprocess for YOLOX - matches mmdetection's test pipeline:
    1. Resize keeping aspect ratio
    2. Pad to square with gray padding (114, 114, 114) — bottom/right only
    3. NO normalization - keep raw pixel values [0-255]
    """
    h, w = image.shape[:2]

    # Resize keeping aspect ratio
    aspect_ratio = w / h
    if h > w:
        new_h = target_size[0]
        new_w = int(np.floor(aspect_ratio * new_h))
    else:
        new_w = target_size[1]
        new_h = int(np.floor(new_w / aspect_ratio))

    img_resized = cv2.resize(image, (new_w, new_h))

    # Pad bottom/right only (matches mmdetection's Pad(pad_to_square=True))
    pad_h = target_size[0] - new_h
    pad_w = target_size[1] - new_w

    img_padded = cv2.copyMakeBorder(
        img_resized, 0, pad_h, 0, pad_w, cv2.BORDER_CONSTANT, value=(114, 114, 114)
    )

    return np.moveaxis(img_padded, -1, 0).astype(np.float32)

In [ ]:
fp32_onnx_predictions_folder = MMDETECTION_OUTPUT_FOLDER / "fp32_onnx_predictions"
fp32_onnx_predictions_folder.mkdir(exist_ok=True)

providers = ["CPUExecutionProvider"]
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

session = ort.InferenceSession(FP32_ONNX_MODEL_PATH, sess_options, providers=providers)

input_meta = session.get_inputs()[0]
input_name = input_meta.name
output_names = [output.name for output in session.get_outputs()]

total_inference_time = 0.0
for image_path in valid_image_paths:
    image = cv2.imread(image_path)  # mmdetection expects bgr images
    image = ensure_3ch_image(image)
    image = preprocess_img_yolox(image, AI_INPUT_IMAGE_SIZE)
    img_batch = np.expand_dims(image, 0)

    start = time.monotonic()
    inputs = {input_name: img_batch}
    outputs = session.run(output_names, inputs)
    total_inference_time += time.monotonic() - start

    vis_image = visualize_bbox(
        image_path=image_path,
        boxes=outputs[0],
        labels=outputs[1],
        scores=outputs[2],
        class_names=classes,
        model_input_size=AI_INPUT_IMAGE_SIZE,
        score_threshold=0.5,
        save_path=fp32_onnx_predictions_folder / image_path.with_suffix(".jpg").name,
    )
print(
    f"Predictions of the FP32 ONNX model were saved to {fp32_onnx_predictions_folder}."
)
print(f"Average inference time: {total_inference_time / len(valid_image_paths)}")

## 3. Quantize ONNX model (optional)
 
This step is required to utilize the NPU at B60, which dramatically speeds up model inference, the drawback is a potential degradation of model quality. This notebook uses a simple MinMax quantization, this works sufficiently well for smaller model such as YOLOX-S and YOLOX-Tiny. 

**Note:** For the best preservation of model quality, especially for larger models (e.g. YOLOX-L, YOLOX-X), we recommend using the Percentile based quantization. Note that the Percentile based quantization requires significantly more time to complete. 

We recommend excluding the NMS from quantization for 2 purposes:
* Preserve postprocessing quality
* Ensure NPU compability

In [ ]:
INT8_ONNX_MODEL_PATH = MODEL_FOLDER / "int8_model.onnx"
NUM_WORKERS = 4
CALIBRATION_SAMPLE_SIZE = 1024  # Please adjust this according to your usecase.

In [ ]:
class SimpleDataset(torch.utils.data.Dataset):
    def __init__(self, image_paths, image_size):
        """
        Args:
            image_paths (list of str): List of file paths to JPEG images
            transform (callable, optional): Optional transform to be applied
        """
        # Filter out non-existent or invalid paths
        self.image_paths = [p for p in image_paths if os.path.exists(p)]
        self.image_size = image_size

        if len(self.image_paths) == 0:
            raise ValueError("No valid image paths found!")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]

        image = cv2.imread(image_path)  # mmdetection expects bgr images
        image = ensure_3ch_image(image)
        image = preprocess_img_yolox(image, AI_INPUT_IMAGE_SIZE)

        return image, 0

In [ ]:
train_images_dir = MMDETECTION_DATA_FOLDER / "train" / "images" / "default"

train_image_paths = [
    p
    for p in train_images_dir.iterdir()
    if p.is_file()
    and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".avif"}
]

calibration_dataset = SimpleDataset(train_image_paths, AI_INPUT_IMAGE_SIZE)

In [ ]:
float32_preprocessed_model_path = Path(
    str(FP32_ONNX_MODEL_PATH).replace(".onnx", "_preprocessed.onnx")
)

print("Quantizing ONNX model")

quant_pre_process(
    input_model_path=FP32_ONNX_MODEL_PATH,
    output_model_path=float32_preprocessed_model_path,
    skip_symbolic_shape=True,
)

# Auto-discover post-processing nodes (bbox decode + NMS) to exclude from
# quantization. This keeps the backbone+neck+head as one large quantized
# subgraph (ideal for NPU) while preserving fp32 precision for bbox
# coordinate math and NMS, which is critical for correct NMS behavior.

postprocess_nodes = find_postprocess_nodes_to_exclude(float32_preprocessed_model_path)
print(f"Excluding {len(postprocess_nodes)} post-processing nodes from quantization")

calibration_kwargs = {
    "dataset": calibration_dataset,
    "batch_size": 1,
    "shuffle": True,
    "num_workers": NUM_WORKERS,
    "persistent_workers": NUM_WORKERS > 0,
}
calibration_data_reader = TorchCalibrationDataReader(
    float32_preprocessed_model_path,
    samples=CALIBRATION_SAMPLE_SIZE,
    **calibration_kwargs,
)

quantize_static(
    model_input=float32_preprocessed_model_path,
    model_output=INT8_ONNX_MODEL_PATH,
    calibration_data_reader=calibration_data_reader,
    quant_format=QuantFormat.QDQ,
    activation_type=QuantType.QUInt8,
    weight_type=QuantType.QInt8,
    per_channel=False,
    calibrate_method=CalibrationMethod.MinMax,
    nodes_to_exclude=postprocess_nodes,
)
print(f"Model quantized and saved to: {INT8_ONNX_MODEL_PATH}")

### 3.1 Compare FP32 and INT8 models

Quantization might lead to a degradation in model quality, it is important to verify the quality of the quantized model by comparing it against the FP32 model. Here we would compare the COCO metrics pre and post quantization. Note that quantization is optional, if the model quality is not acceptable you can always run a FP32 model as long as it fits your inference time requirements.

In [ ]:
int8_onnx_predictions_folder = MMDETECTION_OUTPUT_FOLDER / "int8_onnx_predictions"
int8_onnx_predictions_folder.mkdir(exist_ok=True)

providers = ["CPUExecutionProvider"]
sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

session = ort.InferenceSession(INT8_ONNX_MODEL_PATH, sess_options, providers=providers)

input_meta = session.get_inputs()[0]
input_name = input_meta.name
output_names = [output.name for output in session.get_outputs()]

total_inference_time = 0.0
for image_path in valid_image_paths:
    image = cv2.imread(image_path)  # mmdetection expects bgr images
    image = ensure_3ch_image(image)
    image = preprocess_img_yolox(image, AI_INPUT_IMAGE_SIZE)
    img_batch = np.expand_dims(image, 0)

    start = time.monotonic()
    inputs = {input_name: img_batch}
    outputs = session.run(output_names, inputs)
    total_inference_time += time.monotonic() - start

    vis_image = visualize_bbox(
        image_path=image_path,
        boxes=outputs[0],
        labels=outputs[1],
        scores=outputs[2],
        class_names=classes,
        model_input_size=AI_INPUT_IMAGE_SIZE,
        score_threshold=0.5,
        save_path=int8_onnx_predictions_folder / image_path.with_suffix(".jpg").name,
    )
print(
    f"Predictions of the INT8 ONNX model were saved to {int8_onnx_predictions_folder}."
)
print(f"Average inference time: {total_inference_time / len(valid_image_paths)}")

In [ ]:
ann_file = str(
    MMDETECTION_DATA_FOLDER / "valid" / "annotations" / "instances_default.json"
)
valid_images_dir = MMDETECTION_DATA_FOLDER / "valid" / "images" / "default"

coco_gt = COCO(ann_file)
img_id_to_info = {img["id"]: img for img in coco_gt.dataset["images"]}


def evaluate_onnx_model(model_path, label=""):
    """Run ONNX model on validation set and compute COCO metrics."""
    sess_options = ort.SessionOptions()
    sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    session = ort.InferenceSession(str(model_path), sess_options, providers=providers)
    input_name = session.get_inputs()[0].name
    output_names = [o.name for o in session.get_outputs()]

    target_h, target_w = AI_INPUT_IMAGE_SIZE
    results = []

    for img_id, img_info in img_id_to_info.items():
        img_path = str(valid_images_dir / img_info["file_name"])
        img_bgr = cv2.imread(img_path)
        img_bgr = ensure_3ch_image(img_bgr)
        orig_h, orig_w = img_bgr.shape[:2]

        input_tensor = preprocess_img_yolox(img_bgr, AI_INPUT_IMAGE_SIZE)
        input_batch = np.expand_dims(input_tensor, axis=0)

        boxes, labels, scores = session.run(output_names, {input_name: input_batch})

        # Top-left aligned padding (matches mmdetection)
        scale = min(target_h / orig_h, target_w / orig_w)
        pad_top = 0
        pad_left = 0

        for box, lbl, score in zip(boxes, labels, scores):
            x1 = max(0, (box[0] - pad_left) / scale)
            y1 = max(0, (box[1] - pad_top) / scale)
            x2 = min(orig_w, (box[2] - pad_left) / scale)
            y2 = min(orig_h, (box[3] - pad_top) / scale)
            w, h = x2 - x1, y2 - y1
            if w <= 0 or h <= 0:
                continue
            results.append(
                {
                    "image_id": img_id,
                    "category_id": int(lbl) + 1,
                    "bbox": [float(x1), float(y1), float(w), float(h)],
                    "score": float(score),
                }
            )

    print(f"\n{'='*60}")
    print(f"  {label}: {model_path}")
    print(f"{'='*60}")
    if not results:
        print("  No detections!")
        return None

    coco_dt = coco_gt.loadRes(results)
    coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval.params.maxDets = [100, 300, 1000]  # Match mmdetection's CocoMetric
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    return coco_eval.stats


fp32_stats = evaluate_onnx_model(FP32_ONNX_MODEL_PATH, "FP32")
int8_stats = evaluate_onnx_model(INT8_ONNX_MODEL_PATH, "INT8")

metric_names = [
    "AP @ IoU=0.50:0.95",
    "AP @ IoU=0.50",
    "AP @ IoU=0.75",
    "AP (small)",
    "AP (medium)",
    "AP (large)",
    "AR @ maxDets=100",
    "AR @ maxDets=300",
    "AR @ maxDets=1000",
    "AR (small)",
    "AR (medium)",
    "AR (large)",
]

print(f"\n{'='*60}")
print(f"  FP32 vs INT8 Comparison")
print(f"{'='*60}")
print(f"  {'Metric':<25} {'FP32':>8} {'INT8':>8} {'Δ':>8}")
print(f"  {'-'*25} {'-'*8} {'-'*8} {'-'*8}")
for name, fp32_val, int8_val in zip(metric_names, fp32_stats, int8_stats):
    delta = int8_val - fp32_val
    print(f"  {name:<25} {fp32_val:>8.4f} {int8_val:>8.4f} {delta:>+8.4f}")

## 4. Export ONNX model

### 4.1 YAML format

During model export, a metadata .yaml file is generated, it contains all the necessary information about the model such as the input size, class names, etc. It is essential to use the right parameters to ensure that correctness of the preprocessing. Please review the example metadata file and its description below. For more details, refer to [metadata.md](../metadata.md)

#### 4.1.1 `.yaml` metadata file example

#### 4.1.2 Input preprocessing order

1. (optional) Color space/channel transformation

    As a result of this transformation the image data has shape
    
        a. (batch_size, original_height, original_width, channels) for channel_order=NHWC
      
        b. (batch_size, channels, original_height, original_width) for channel_order=NCHW

   where channels is 1 or 3 and the image channel data matches the order defined by color_space (RGB/BGR/GRAYSCALE).

3. Image resize

4. (optional) Unit scaling.

    `x = x / 255`

   
5. (optional) Standardization

   `x = (x - standardization_mean) / standardization_std`

### 4.2 Export INT8 model

In [ ]:
UNIVISION_INT8_MODEL_PATH = MODEL_FOLDER / "example_int8_model.u3o"

In [ ]:
if any(is_rgb_image(path) for path in valid_image_paths):
    dataset_color_mode = DatasetColorMode.COLOR
else:
    dataset_color_mode = DatasetColorMode.MONOCHROME

dataset_color_mode

In [ ]:
input_example = read_and_resize_input_example(
    valid_image_paths[0], AI_INPUT_IMAGE_SIZE, dataset_color_mode
)

In [ ]:
export_univision_model_v3(
    univision_model_path=UNIVISION_INT8_MODEL_PATH,
    onnx_model_path=INT8_ONNX_MODEL_PATH,
    classes=classes,
    input_example=input_example,
    model_name=None,
    model_uuid=str(uuid.uuid4()),
    inference_device="AUTO",
    quantization="INT8",
    resize_mode="FIT_WITH_PADDING",
    resize_padding_value=(114, 114, 114),
    resize_image_alignment_horizontal="LEFT",
    resize_image_alignment_vertical="TOP",
    unit_scaling=False,
    standardization_std=None,
    standardization_mean=None,
    channel_order="NCHW",
    dataset_color_mode=dataset_color_mode,
    input_color_space="BGR",
    output_type="OBJECT_DETECTION",
    class_thresholds=[0.5] * len(classes),
    boxes_output_index=0,
    labels_output_index=1,
    scores_output_index=2,
    boxes_format="left_top_right_bottom",
    boxes_coordinates="absolute",
    max_detections=20,
)